In [1]:
# Importing necessary libraries
import pandas as pd

In [2]:
# Loading Dataset
movies = pd.read_csv(filepath_or_buffer="../data/movies.csv").loc[:, ["original_title", "overview"]]
movies.head()

,original_title,overview
0,Avatar,"In the 22nd century, a paraplegic Marine is di..."
1,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha..."
2,Spectre,A cryptic message from Bond’s past sends him o...
3,The Dark Knight Rises,Following the death of District Attorney Harve...
4,John Carter,"John Carter is a war-weary, former military ca..."


In [3]:
# Preprocessing 1 - Lowercasing
movies['overview'] = movies['overview'].str.lower()
movies.head()

,original_title,overview
0,Avatar,"in the 22nd century, a paraplegic marine is di..."
1,Pirates of the Caribbean: At World's End,"captain barbossa, long believed to be dead, ha..."
2,Spectre,a cryptic message from bond’s past sends him o...
3,The Dark Knight Rises,following the death of district attorney harve...
4,John Carter,"john carter is a war-weary, former military ca..."


In [4]:
# Preprocessing 2 - Removing HTML Tags
import re

# Incase there are null values
movies.dropna(inplace=True)

def remove_html_tags(text):
    pattern = re.compile("<.*?>")
    return pattern.sub(r'', text)

movies['overview'] = movies['overview'].apply(remove_html_tags)
movies.sample(5)

,original_title,overview
1889,He Got Game,a basketball player's father must try to convi...
2059,Molière,"molière, a down-and-out actor-cum-playwright u..."
1827,Teenage Mutant Ninja Turtles II: The Secret of...,the turtles and the shredder battle once again...
79,Iron Man 2,with the world now aware of his dual life as t...
3611,A Farewell to Arms,british nurse catherine barkley (helen hayes) ...


In [5]:
# Preprocessing 3 - Removing Links
def removing_links(text):
    pattern = re.compile(r'https?://\S+|www\.\S+')

    # This pattern is useful to filterout all types of links
    '''
        text1 = 'Check out my notebook https://www.kaggle.com/campusx/notebook8223fc1abb'
        text2 = 'Check out my notebook http://www.kaggle.com/campusx/notebook8223fc1abb'
        text3 = 'Google search here www.google.com'
        text4 = 'For notebook click https://www.kaggle.com/campusx/notebook8223fc1abb to search check www.google.com'
    '''

    return pattern.sub(r'', text)

movies['overview'] = movies['overview'].apply(removing_links)
movies.sample(5)

,original_title,overview
3346,Jumping the Broom,two very different families converge on martha...
3276,It's Kind of a Funny Story,a clinically depressed teenager gets a new sta...
2643,Возвращение,a story of two russian boys whose father sudde...
2626,Idle Hands,anton is a cheerful but exceedingly non-ambiti...
366,Hollow Man,"cocky researcher, sebastian caine is working o..."


In [6]:
from urlextract import URLExtract # Incase you have'nt installed - !pip install urlextract

extractor = URLExtract()
urls = extractor.find_urls("Let's have URL www.stackoverflow.com as an example.")
print(urls) # prints: ['stackoverflow.com']

['www.stackoverflow.com']


In [7]:
# Preprocessing 4 - Remove Punctuations
import string
exclude = string.punctuation
print(exclude)

def remove_punctuation(overview):
    return "".join([(t if t not in exclude else "") for t in overview])
    # return text.translate(str.maketrans('', '', exclude)) - maketrans is a mapping function

movies['overview'] = movies['overview'].apply(remove_punctuation)
movies.sample(5)

!"#$%&'()*+,-./:;<=>?@[\]^_`{|}~


,original_title,overview
4612,Old Joy,two old pals reunite for a camping trip in ore...
2907,Pet Sematary,dr louis creeds family moves into the country ...
4314,Crowsnest,in late summer of 2011 five young friends on a...
3391,Dom Hemingway,after spending 12 years in prison for keeping ...
4375,Blue Car,gifted 18yearold meg has been abandoned by her...


In [8]:
# Preprocessing 5 - Chat Words Treatment
import json

with open(file="../data/chat_words_dictonary.json") as f:
    chat_words = json.load(f)

for index, (key, value) in enumerate(chat_words.items()):
    if index == 10: break
    print(f"{key} : {value}")

atm : at this moment
brb : be right back
lol : laughing out loud
btw : by the way
g2g : got to go
idk : I don't know
ikr : I know, right
np : no problem
omg : oh my god
rofl : rolling on floor laughing


In [9]:
def chat_words_treatment(text):
    return " ".join(list(map(lambda t: t if t not in chat_words else chat_words[t], text.split(" "))))

movies['overview'] = movies['overview'].apply(chat_words_treatment)
movies.sample(5)

,original_title,overview
2431,My Life in Ruins,a greek tour guide named georgia attempts to r...
3689,All Hat,an excon returns to his rural ontario roots an...
3226,Namastey London,indianborn manmohan malhotra decided to reloca...
1135,Lord of War,yuri orlov is a globetrotting arms dealer and ...
1242,The Reaping,katherine morrissey a former christian mission...


In [ ]:
# Preprocessing 6 - Spelling Correction
from textblob import TextBlob

def spelling_correction(text):

    textBlb = TextBlob(text) # Creating TextBlob object
    return textBlb.correct().string # Correcting spelling

spelling_correction("Hellow myi nmame ins dexter morgan")
# Note: Even for small datasets it takes lot of time

In [ ]:
# Preprocessing 7 - Removing Stop-Words
import nltk
nltk.download('stopwords')

from nltk.corpus import stopwords
stopwords = stopwords.words('english')

def remove_stopwords(text):
    result = []
    for word in text.split(" "):
        if word not in stopwords:
            result.append(word)
    return " ".join(result)

movies['Description'] = movies['Description'].apply(remove_stopwords)

In [15]:
# Preprocessing 8 - Removing Emojis
def remove_emoji(text):
    emoji_pattern = re.compile("["
            u"\U0001F600-\U0001F64F"  # emoticons
            u"\U0001F300-\U0001F5FF"  # symbols & pictographs
            u"\U0001F680-\U0001F6FF"  # transport & map symbols
            u"\U0001F1E0-\U0001F1FF"  # flags (iOS)
            u"\U00002702-\U000027B0"
            u"\U000024C2-\U0001F251"
        "]+", 
        flags=re.UNICODE)
    return emoji_pattern.sub(r'', text)

remove_emoji("Loved the movie. It was 😘😘")

# movies['Description'] = movies['Description'].apply(remove_emoji)

'Loved the movie. It was '

In [ ]:
# Preprocessing 8' - Replacing emojis with there english representation
import emoji

print(emoji.demojize('Python is 🔥'))
print(emoji.demojize('Loved the movie. It was 😘'))
print(emoji.demojize('Loved the movie. It was 😂'))

Python is :fire:
Loved the movie. It was :face_blowing_a_kiss:
Loved the movie. It was :face_with_tears_of_joy:
